# Clustering Membership Tracking Setup 

1) Solve the label switching problem
2) Quantify the stability/ change using a transition metric

**Dealing with label switching** 
1) compute a similarity matrix, a k x k matrix where rows are clusrers at time t and the columns are clusters at time t + 1
2) Fill each cell with the distance metric between the clusters at t and t + 1
- centroid distance 
- set intersection (maximise the overalap of points)
3) Optimal macthing - run the hungarian algorithm on this matrix 


**Measuring instabilibty/change**
Metric 1: Tracking size distribution changes (Macro level)
- If only care about the sizes of clusters changing (can treat the cluster sizes as a probability distribution and measure the distance between them)
- EMD / wassersetein
- KL divergence / jensen shannon divergence
- Can use OT (with the ground cost as the distance between centroids)

To track indiviual users, we can run the hungarian algorithm with the jacard so that we know which cluster user x was in at time t and time t + 1. so we can then compute a metric based on this (using proportions).

Here we are just looking at how the points are moving between clusters.

For ARI, a single point changing clusters impacts way more pairs in a large dataset than in a small one.

## Data & Packages

In [1]:
# Main packages 
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

# Clustering packages
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score,adjusted_rand_score

# Optimisation packages
from scipy.optimize import linear_sum_assignment

# Parallel processing packages
from joblib import Parallel, delayed

In [2]:
# Load data
df = pl.scan_csv("/home/lanl/data/cyber1/auth.txt.gz",has_header=False,separator=",",
                 new_columns= ['time','src_user','dest_user','src_comp','dest_comp',
                               'auth_type','logon_type','auth_orientation','outcome'])

In [3]:
# Keep only human users
df = df.filter(pl.col("src_user").str.starts_with("U"))

## Functions & Global info

In [4]:
# Time conversions
seconds_in_day = 60 * 60 * 24

# Time aggregation
agg_hour_level = 1

# Number of buckets in a week of data
buckets_per_week = (7 * 24) // agg_hour_level

# Silhouette sample size 
sample_size = 1000

In [5]:
# FUNCTION - build features dataframe
def build_features(df,agg_hour_level):

    agg_seconds = agg_hour_level * 60 * 60

    return (
        df.with_columns(
            bucket = pl.col('time') // agg_seconds,
            theta = ((pl.col('time') % seconds_in_day)/ seconds_in_day) * 2 * np.pi,
            is_failure = (pl.col('outcome') == 'Fail').cast(pl.Int8),
        )
        .group_by(['src_user','bucket'])
        .agg(
            n_events = pl.len(),
            failure_ratio = pl.col('is_failure').mean(),
            n_distinct_dest = pl.col('dest_comp').n_unique(),
            n_distinct_src = pl.col('src_comp').n_unique(),
            c_bar = pl.col('theta').cos().mean(),
            s_bar = pl.col('theta').sin().mean(),
        )
        .with_columns(
             log_n_events=pl.col("n_events").log1p(),
             log_n_distinct_dest=pl.col("n_distinct_dest").log1p(),
             log_n_distinct_src=pl.col("n_distinct_src").log1p(),
        )
        .collect()
    )

In [6]:
# Relevant feauture columns
feature_cols = [
    "log_n_events",
    "failure_ratio",
    "log_n_distinct_dest",
    "log_n_distinct_src",
    "c_bar",
    "s_bar",
]

In [7]:
# FUNCTION - process features for clustering 
def cluster_preprocess(features_df,feature_cols,week):

    lb = (week - 1) * buckets_per_week
    ub = lb + buckets_per_week - 1

    features_week = features_df.filter(pl.col('bucket').is_between(lb,ub))

    X = features_week.select(feature_cols).to_numpy()
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    return features_week, X_scaled

In [8]:
# FUNCTION - kmeans 
def fit_kmeans(k, Y, sample_size):
    km = KMeans(n_clusters=k, random_state=123, n_init=10)
    labels = km.fit_predict(Y)
    
    sil = silhouette_score(Y, labels, sample_size=sample_size, random_state=123)
    ch  = calinski_harabasz_score(Y, labels)   
    db  = davies_bouldin_score(Y, labels)    
    
    return k, sil,ch,db

In [ ]:
# Create features dataframe
features_df = build_features(df,agg_hour_level)

## Clustering

In [ ]:
# Cluster for each week
n_weeks = 8
k = 3
weekly_results = {}

for week in range(1, n_weeks + 1):

    features_week, X_scaled = cluster_preprocess(features_df, feature_cols, week=week)

    km = KMeans(n_clusters=k, random_state=123, n_init=10) 
    labels = km.fit_predict(X_scaled)   

    features_week = (features_week.with_columns(pl.Series("cluster", labels)).select(['src_user','bucket','cluster']))

    weekly_results[week] = features_week

## Jacard & Hungarian

In [ ]:
JACARD = np.zeros((n_weeks - 1,k,k))

for week in range(1,n_weeks):
    
    w_curr = weekly_results[week].with_columns(
        relative_bucket=pl.col('bucket') % buckets_per_week
    )

    w_next = weekly_results[week + 1].with_columns(
        relative_bucket=pl.col('bucket') % buckets_per_week
    )

    overlap = w_curr.join(w_next, on=['src_user', 'relative_bucket'], how='inner', suffix='_next')

    labels_curr = overlap['cluster'].to_numpy()
    labels_next = overlap['cluster_next'].to_numpy()

    for i in range(k):
        mask_i = (labels_curr == i)
        
        for j in range(k):
            mask_j = labels_next == j
            intersection = np.sum(mask_i & mask_j)
            union = np.sum(mask_i | mask_j)
            JACARD[week - 1, i,j] = intersection/union 

    _, col_ind = linear_sum_assignment(JACARD[week - 1], maximize=True)

    mapping = {col_ind[c]: c for c in range(k)}
    labels_next = np.array([mapping[label] for label in labels_next])

    print(np.unique(labels_curr, return_counts = True))
    print(np.unique(labels_next, return_counts = True))

(array([0, 1, 2], dtype=int32), array([  8328, 388744, 218552]))
(array([0, 1, 2]), array([  8600, 386693, 220331]))
(array([0, 1, 2], dtype=int32), array([229063, 389431,   8030]))
(array([0, 1, 2]), array([194915, 255085, 176524]))
(array([0, 1, 2], dtype=int32), array([178539, 197826, 257312]))
(array([0, 1, 2]), array([  8087, 234892, 390698]))
(array([0, 1, 2], dtype=int32), array([263266,   8997, 438142]))
(array([0, 1, 2]), array([263880,   9114, 437411]))
(array([0, 1, 2], dtype=int32), array([ 11061, 436290, 262132]))
(array([0, 1, 2]), array([ 11293, 437668, 260522]))
(array([0, 1, 2], dtype=int32), array([414063, 253437,  16373]))
(array([0, 1, 2]), array([432156, 235232,  16485]))
(array([0, 1, 2], dtype=int32), array([226796,  22846, 415595]))
(array([0, 1, 2]), array([229382,  23296, 412559]))


## Membership Tracking

The first simple way to track is put the clusters in a 3 x 3 matrix and compute the trace proportions, so the percentage that stayed in cluster 0, 1,& 2. 

In [ ]:
cluster_1 = np.array([385846, 221449, 8329])
cluster_2 = np.array([387053, 219971, 8600])

n = np.sum(cluster_1)

print((n - (np.abs(cluster_1[0] - cluster_2[0]) + np.abs(cluster_1[1] - cluster_2[1]) + np.abs(cluster_1[2] - cluster_2[2])))/n)


cluster_1 = np.array([8030, 389795, 228699])
cluster_2 = np.array([7934, 420764, 197826])

n = np.sum(cluster_1)

print((n - (np.abs(cluster_1[0] - cluster_2[0]) + np.abs(cluster_1[1] - cluster_2[1]) + np.abs(cluster_1[2] - cluster_2[2])))/n)

0.9951983678349122
0.9011402595910133


# Reading

MONIC: modeling and monitoring cluster transitions

https://dl.acm.org/doi/abs/10.1145/1150402.1150491

**Abstract**

Anoamly detection is field where anomaly detection is relevant. It is necessary to provide insights about the nature of cluster change. Is a cluster corresponding to a group of customers simply disappearing or are its members migrating to other clusters.

Is a new emerging cluster reflecting a new target group of customers or does it rather consist of existing customers whose preferences shift?


**Intro**

Clusters upon the data of real applications are affected by changes in the underlying population. There is plenty of research in adapting the clusters to the changed population. Tracing and understanding the changes themselves, can give insight on the populaiton and the supporting strategic decisions.

**Related work**

FOCUS - determines whether the underlying structure of clusters has shifted over time or across different data sources.

PANDA - While FOCUS forces new data into a single, rigid "old" model, PANDA is designed to compare two entirely different, independently generated models (like two different clusterings from two different time periods).

PAM - models patterns as temporal objects and captures their evolution, focusing mainly on association rules.

MONIC builds upon the insights of PANDA and PAM.

**Cluster Matching**

(Cluster overlap). Let $\zeta_i,\zeta_j$ ($i<j$) be the clusterings at $t_i,t_j$, and let $X\in\zeta_i$, $Y\in\zeta_j$ be two clusters. The \emph{overlap of $X$ to $Y$} is the normalized sum of weights of the records in their set intersection:

$\operatorname{overlap}(X,Y)=\frac{\sum_{a\in X\cap Y}\operatorname{age}(a,t_j)}{\sum_{x\in X}\operatorname{age}(x,t_j)}.$

Definition 4 (Cluster match). Let $X$ be a cluster in the clustering $\zeta_i$ at $t_i$ and $Y$ be a cluster in $\zeta_j$ at $t_j>t_i$. Further, let $\tau\equiv\tau_{\mathrm{match}}\in[0.5,1]$ be a threshold value. $Y$ is \emph{a match for $X$ in $\zeta_j$ subject to $\tau$}, i.e. $Y=\operatorname{match}_{\tau}(X,\zeta_j)$, if and only if $Y\in\zeta_j$ is the cluster with the maximum overlap for $X$ and the overlap of $X$ to $Y$ is at least $\tau$:

$\operatorname{overlap}(X,Y)=\max_{U\in\zeta_j}\{\operatorname{overlap}(X,U)\}\ge\tau.$

If there is no such $Y\in\zeta_j$, then $\operatorname{match}_{\tau}(X,\zeta_j)=\varnothing$.

**but for same number of clusters then we can take the jacard index for overlap**


## Hungarian algorithm - methodological precidence : 
Lange, Roth, Braun, and Buhmann (2004), "Stability-Based Validation of Clustering Solutions," 